In [ ]:
# Cell 1 - check the GPU actually works
import torch

print("torch", torch.__version__)
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU T4 x2, then restart the session.")
print("GPU:", torch.cuda.get_device_name(0))

# Run a real convolution on the GPU. If this torch build can't drive the card
# (e.g. an old P100), it fails here in seconds instead of minutes later.
x = torch.randn(8, 3, 224, 224, device="cuda")
w = torch.randn(16, 3, 3, 3, device="cuda")
y = torch.nn.functional.conv2d(x, w)
torch.cuda.synchronize()
print("GPU works, test output shape:", tuple(y.shape))

In [ ]:
# Cell 2 - helper, code, timm
import os
import subprocess


def run(cmd):
    """Run a shell command, show its output live, and STOP the notebook if it fails.

    A plain `!command` ignores failures, so a broken smoke test would still
    carry on into the 4-hour run. This raises instead.
    """
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, executable="/bin/bash")
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed (exit code {p.returncode}): {cmd}")


REPO = "/tmp/repo"
run(f"rm -rf {REPO} && git clone -q -b feature/train-resume --depth 1 "
    f"https://github.com/Akash7762/Amazon-Vision-Based-Price-Predictor.git {REPO}")
os.chdir(REPO)
run("git log --oneline -1")
run("grep -q -- '--max-hours' model/train.py && echo 'code is up to date' "
    "|| (echo 'OLD CODE on GitHub: run git push on your PC first' && exit 1)")

run('pip install -q "timm>=1.0.27"')
# pip must not have swapped Kaggle's GPU torch for a CPU-only one
run('python -c "import torch, timm; assert torch.cuda.is_available(); '
    'print(\'torch\', torch.__version__, \'| timm\', timm.__version__, \'| cuda OK\')"')

In [ ]:
# Cell 3 - point at the data and check it
import glob
import os
import pandas as pd

META = "/kaggle/input/datasets/akashverma7762/amazon-price-metadata/metadata.csv"
IMG = "/kaggle/input/datasets/akashverma7762/amazon-price-images-v2/images"

if not (os.path.exists(META) and os.path.isdir(IMG)):
    # Fallback search. It only looks at dataset-folder level and never lists the
    # 73k-image folder itself, which is what made the old search take minutes.
    metas = glob.glob("/kaggle/input/*/metadata.csv") + glob.glob("/kaggle/input/datasets/*/*/metadata.csv")
    imgs = glob.glob("/kaggle/input/*/images") + glob.glob("/kaggle/input/datasets/*/*/images")
    print("found metadata:", metas)
    print("found images:  ", imgs)
    assert len(metas) == 1 and len(imgs) == 1, (
        "Attach exactly two inputs: amazon-price-metadata and amazon-price-images-v2 "
        "(not the old amazon-price-images).")
    META, IMG = metas[0], imgs[0]

ids = set(pd.read_csv(META)["sample_id"].astype(str))
have = {f[:-4] for f in os.listdir(IMG) if f.endswith(".jpg")}  # takes ~1 min, the folder is big
print(f"META = {META}")
print(f"IMG  = {IMG}")
print(f"images={len(have)} missing={len(ids - have)}")
assert len(ids - have) < 100, "Images missing - wrong images dataset attached?"

manifest = os.path.join(IMG, "_preprocessing.json")
print(open(manifest).read() if os.path.exists(manifest) else "WARNING: no _preprocessing.json")

In [ ]:
# Cell 4 - smoke test: 5 batches, ~1-2 min. Catches problems before the long run.
run(f"python -u model/train.py --subset-mode full --metadata {META} --image-dir {IMG} "
    f"--epochs 1 --max-batches 5 --batch-size 64 --num-workers 4 "
    f"--checkpoint-dir /tmp/smoke --log-path /tmp/smoke/smoke.log")

In [ ]:
# Cell 5 - the real training run (~4 hours on a T4)
# --max-hours 8: stops cleanly before an epoch that would go past 8 hours, so the
# run always finishes and saves its output instead of being killed by Kaggle.
EPOCHS = 15
run(f"python -u model/train.py --subset-mode full --metadata {META} --image-dir {IMG} "
    f"--epochs {EPOCHS} --batch-size 64 --lr 1e-4 --num-workers 4 --max-hours 8 "
    f"--checkpoint-dir /kaggle/working/checkpoints --log-path /kaggle/working/logs/train.log")

In [ ]:
# Cell 6 - summary, trim checkpoints, plot the loss curve
import glob
import json
import os
import re
import matplotlib.pyplot as plt

CK = "/kaggle/working/checkpoints"
BASELINE = 14.03  # val loss if you always predict the median train price ($14)

h = json.load(open(f"{CK}/history.json"))
best = min(h, key=lambda r: r["val_loss"])
print(f"epochs done: {h[-1]['epoch']} of {EPOCHS}")
print(f"best epoch:  {best['epoch']}   val loss {best['val_loss']:.3f}   (median baseline {BASELINE})")
if h[-1]["epoch"] < EPOCHS:
    print(f"Stopped early by the time budget - resume from epoch_{h[-1]['epoch']}.pt")

# Each checkpoint is ~330 MB. Keep best.pt and the newest epoch_N.pt (for resuming).
done = sorted(int(re.search(r"epoch_(\d+)\.pt$", p).group(1)) for p in glob.glob(f"{CK}/epoch_*.pt"))
for e in done[:-1]:
    os.remove(f"{CK}/epoch_{e}.pt")

ep = [r["epoch"] for r in h]
plt.figure(figsize=(7, 4))
plt.plot(ep, [r["train_loss"] for r in h], marker="o", label="train")
plt.plot(ep, [r["val_loss"] for r in h], marker="o", label="val")
plt.axhline(BASELINE, ls="--", c="gray", label="median-price baseline")
plt.xlabel("epoch"); plt.ylabel("SmoothL1 loss"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig("/kaggle/working/loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nSaved output:")
for f in sorted(glob.glob("/kaggle/working/**", recursive=True)):
    if os.path.isfile(f):
        print(f"{os.path.getsize(f) / 1e6:8.1f} MB  {f}")
print(json.dumps(h, indent=1))